# Camino euleriano

Un camino euleriano recorre todas las aristas exactamente una vez; si empieza y acaba en el mismo nodo es un circuito. En un grafo no dirigido conexo existe circuito si todos los nodos tienen grado par, y camino si hay exactamente dos de grado impar (los extremos). En uno dirigido, si cada nodo tiene igual grado de entrada que de salida (o uno con una salida de más y otro con una entrada de más). El algoritmo de Hierholzer lo construye en $O(E)$.

## 1. Los puentes de Königsberg

Comprobar con los grados por qué no se pueden cruzar los siete puentes de Königsberg una sola vez cada uno.

In [1]:
import random
from collections import Counter, defaultdict
from itertools import permutations

puentes = [("Norte", "Isla"), ("Norte", "Isla"), ("Sur", "Isla"), ("Sur", "Isla"),
           ("Norte", "Este"), ("Sur", "Este"), ("Isla", "Este")]
grado = Counter()
for u, v in puentes:
    grado[u] += 1
    grado[v] += 1
impares = [u for u in grado if grado[u] % 2]
print(dict(grado))
print("nodos de grado impar:", impares, "-> hay camino" if len(impares) in (0, 2) else "-> no hay camino ni circuito")

{'Norte': 3, 'Isla': 5, 'Sur': 3, 'Este': 3}
nodos de grado impar: ['Norte', 'Isla', 'Sur', 'Este'] -> no hay camino ni circuito


## 2. Hierholzer

Implementar Hierholzer para grafos no dirigidos con aristas repetidas (multigrafo): partir de un nodo impar si lo hay y avanzar quitando aristas; al quedarse sin salida, añadir el nodo al camino. Probarlo añadiendo un puente a Königsberg.

In [2]:
def camino_euleriano(aristas):
    adyacentes = defaultdict(list)
    for i, (u, v) in enumerate(aristas):
        adyacentes[u].append((v, i))
        adyacentes[v].append((u, i))
    impares = [u for u in adyacentes if len(adyacentes[u]) % 2]
    if len(impares) not in (0, 2):
        return None
    usada, pila, camino = [False] * len(aristas), [impares[0] if impares else aristas[0][0]], []
    while pila:
        u = pila[-1]
        while adyacentes[u] and usada[adyacentes[u][-1][1]]:
            adyacentes[u].pop()
        if adyacentes[u]:
            v, i = adyacentes[u].pop()
            usada[i] = True
            pila.append(v)
        else:
            camino.append(pila.pop())
    return camino[::-1] if len(camino) == len(aristas) + 1 else None  # None si no es conexo

print(camino_euleriano(puentes))
con_puente_nuevo = puentes + [("Norte", "Sur")]
print(camino_euleriano(con_puente_nuevo))

None
['Isla', 'Este', 'Sur', 'Norte', 'Isla', 'Sur', 'Isla', 'Norte', 'Este']


## 3. Contra fuerza bruta

Comprobar que el camino usa cada arista una vez y que existe exactamente cuando la búsqueda exhaustiva sobre el orden de las aristas encuentra uno, en multigrafos aleatorios pequeños.

In [3]:
def es_camino(camino, aristas):
    usadas = Counter(frozenset(p) for p in zip(camino, camino[1:]))
    return usadas == Counter(frozenset(a) for a in aristas)

def existe_bruto(aristas):
    def busca(u, restantes):
        if not restantes:
            return True
        for i, (a, b) in enumerate(restantes):
            if u in (a, b) and busca(b if u == a else a, restantes[:i] + restantes[i + 1:]):
                return True
        return False
    nodos = {x for a in aristas for x in a}
    return any(busca(u, aristas) for u in nodos)

random.seed(1)
for _ in range(400):
    n = random.randint(2, 5)
    aristas = [tuple(random.sample(range(n), 2)) for _ in range(random.randint(1, 7))]
    c = camino_euleriano(aristas)
    assert (c is not None) == existe_bruto(aristas)
    if c:
        assert es_camino(c, aristas)
print("ok")

ok


## 4. Secuencia de De Bruijn

Una secuencia de De Bruijn de orden $k$ sobre un alfabeto contiene cada palabra de longitud $k$ exactamente una vez (de forma circular). Se obtiene con un circuito euleriano dirigido en el grafo cuyos nodos son las palabras de longitud $k - 1$. Usarla para abrir una caja fuerte de 3 dígitos binarios y de 2 dígitos decimales probando el mínimo de teclas.

In [4]:
from itertools import product

def de_bruijn(alfabeto, k):
    salidas = {"".join(p): list(alfabeto) for p in product(alfabeto, repeat=k - 1)}
    pila, circuito = [alfabeto[0] * (k - 1)], []
    while pila:
        u = pila[-1]
        if salidas[u]:
            c = salidas[u].pop()
            pila.append(u[1:] + c)
        else:
            circuito.append(pila.pop())
    circuito.reverse()
    return circuito[0] + "".join(nodo[-1] for nodo in circuito[1:])

for alfabeto, k in [("01", 3), ("0123456789", 2)]:
    s = de_bruijn(alfabeto, k)
    palabras = {s[i:i + k] for i in range(len(s) - k + 1)}
    assert palabras == {"".join(p) for p in product(alfabeto, repeat=k)}
    print(f"orden {k} sobre {len(alfabeto)} símbolos: {len(s)} teclas" + (f" -> {s}" if len(s) < 20 else ""))

orden 3 sobre 2 símbolos: 10 teclas -> 0011101000
orden 2 sobre 10 símbolos: 101 teclas
